In [1]:
import geopandas as gpd


PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\FEWSNET_IPC\FEWS NET Admin Boundaries\FEWS_Admin_LZ_v3.shp'

gdf = gpd.read_file(PATH)

# see column names
print(gdf.columns)

Index(['cov_start', 'cov_end', 'report_mon', 'unit_name', 'ADMIN0', 'ADMIN1',
       'ADMIN2', 'ADMIN3', 'LZCODE', 'LZNAME', 'admin_code', 'admin_name',
       'ISO', 'adm0_name', 'AREA2', 'AREA1', 'geometry'],
      dtype='object')


In [2]:
# read IPC,C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\IPC_2017_2025\All_IPC_classification_2017-2025.geojson
import geopandas as gpd
PATH_IPC = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\IPC_2017_2025\All_IPC_classification_2017-2025.geojson'

gdf_ipc = gpd.read_file(PATH_IPC)

In [3]:
# read CH all data
PATH_CH_all = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\IPCCH\cadre_harmonise_caf_ipc_may25.xlsx'
PATH_CH_SHAPE = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\IPCCH\wca_chipc_current_march2023.geojson'

#read csv
import pandas as pd
df_ch = pd.read_excel(PATH_CH_all, sheet_name='Sheet1')
gdf_ch_shape = gpd.read_file(PATH_CH_SHAPE)


In [4]:
# keep id, estimated_population, population, from, to, anl_id, title, overall_phase, country, year, phase3_worse_population, phase1_population, phase2_population, phase3_population, phase4_population, phase5_population, geometry

gdf_ipc_sub = gdf_ipc[['id', 'estimated_population', 'population', 'from', 'to', 'anl_id', 'title', 'overall_phase', 'country', 'year', 'phase3_worse_population', 'phase1_population', 'phase2_population', 'phase3_population', 'phase4_population', 'phase5_population', 'geometry']]

In [5]:
# for df_ch, first filter chtype==current
df_ch= df_ch[df_ch['chtype'] == 'current']

In [6]:
# reindex df_ch
df_ch = df_ch.reset_index(drop=True)

In [7]:
#  add an id column to df_ch
df_ch['id'] = df_ch.index + 1

In [8]:
# for gdf_ch_shape_2 and gdf_ch_shape, keep only admin2_pcod2 and geometry, then unique on admin2_pcod2 and append these two geodataframe into one gdf
gdf_ch_shape_sub = gdf_ch_shape[['adm3_pcod2', 'geometry']].drop_duplicates(subset=['adm3_pcod2'])


# drop if adm2_pcod2 is missing
gdf_ch_shape_combined = gdf_ch_shape_sub.dropna(subset=['adm3_pcod2'])


# keep adm0_name, adm2_name, adm2_pcod2, reference_year, reference_label, population, phase_class, phase1, phase2, phase3, phase4, phase5, phase35
df_ch_sub = df_ch[['id','adm0_name', 'adm2_name', 'adm3_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# drop if adm2_pcod2 is missing
df_ch_sub = df_ch_sub.dropna(subset=['adm3_pcod2'])


# merge df_ch_sub with gdf_ch_shape_combined on adm2_pcod2
gdf_ch_merged = df_ch_sub.merge(gdf_ch_shape_combined, on='adm3_pcod2', how='left', indicator=True)


In [9]:

# gdf_ch_merged indicator
print(gdf_ch_merged['_merge'].value_counts())


_merge
both          107
left_only      63
right_only      0
Name: count, dtype: int64


In [10]:
gdf_ch_merged = gdf_ch_merged[gdf_ch_merged['_merge'] != 'left_only'].drop(columns=['_merge'])

In [11]:
df_ch_remaining = df_ch[~df_ch['id'].isin(gdf_ch_merged['id'])]

In [12]:
# for gdf_ch_shape_2 and gdf_ch_shape, keep only admin2_pcod2 and geometry, then unique on admin2_pcod2 and append these two geodataframe into one gdf
gdf_ch_shape_sub = gdf_ch_shape[['adm2_5_pcod2', 'geometry']].drop_duplicates(subset=['adm2_5_pcod2'])


# drop if adm2_pcod2 is missing
gdf_ch_shape_combined = gdf_ch_shape_sub.dropna(subset=['adm2_5_pcod2'])


# keep adm0_name, adm2_name, adm2_pcod2, reference_year, reference_label, population, phase_class, phase1, phase2, phase3, phase4, phase5, phase35
df_ch_sub = df_ch_remaining[['id','adm0_name', 'adm2_name', 'adm2_5_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# drop if adm2_pcod2 is missing
df_ch_sub = df_ch_sub.dropna(subset=['adm2_5_pcod2'])


# merge df_ch_sub with gdf_ch_shape_combined on adm2_pcod2
gdf_ch_merged_2 = df_ch_sub.merge(gdf_ch_shape_combined, on='adm2_5_pcod2', how='left', indicator=True)

In [13]:

# drop left_only
gdf_ch_merged_2 = gdf_ch_merged_2[gdf_ch_merged_2['_merge'] != 'left_only'].drop(columns=['_merge'])

In [14]:
df_ch_remaining = df_ch_remaining[~df_ch_remaining['id'].isin(gdf_ch_merged_2['id'])]

In [15]:
# for gdf_ch_shape_2 and gdf_ch_shape, keep only admin2_pcod2 and geometry, then unique on admin2_pcod2 and append these two geodataframe into one gdf
gdf_ch_shape_sub = gdf_ch_shape[['adm2_pcod2', 'geometry']].drop_duplicates(subset=['adm2_pcod2'])


# drop if adm2_pcod2 is missing
gdf_ch_shape_combined = gdf_ch_shape_sub.dropna(subset=['adm2_pcod2'])


# keep adm0_name, adm2_name, adm2_pcod2, reference_year, reference_label, population, phase_class, phase1, phase2, phase3, phase4, phase5, phase35
df_ch_sub = df_ch_remaining[['id','adm0_name', 'adm2_name', 'adm2_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# drop if adm2_pcod2 is missing
df_ch_sub = df_ch_sub.dropna(subset=['adm2_pcod2'])


# merge df_ch_sub with gdf_ch_shape_combined on adm2_pcod2
gdf_ch_merged_3 = df_ch_sub.merge(gdf_ch_shape_combined, on='adm2_pcod2', how='left', indicator=True)

In [16]:
gdf_ch_merged_3 = gdf_ch_merged_3[gdf_ch_merged_3['_merge'] != 'left_only'].drop(columns=['_merge'])

In [17]:
df_ch_remaining = df_ch_remaining[~df_ch_remaining['id'].isin(gdf_ch_merged_3['id'])]

In [18]:
# for gdf_ch_shape_2 and gdf_ch_shape, keep only admin2_pcod2 and geometry, then unique on admin2_pcod2 and append these two geodataframe into one gdf
gdf_ch_shape_sub = gdf_ch_shape[['adm1_5_pcod2', 'geometry']].drop_duplicates(subset=['adm1_5_pcod2'])


# drop if adm2_pcod2 is missing
gdf_ch_shape_combined = gdf_ch_shape_sub.dropna(subset=['adm1_5_pcod2'])


# keep adm0_name, adm2_name, adm2_pcod2, reference_year, reference_label, population, phase_class, phase1, phase2, phase3, phase4, phase5, phase35
df_ch_sub = df_ch_remaining[['id','adm0_name', 'adm1_name', 'adm1_5_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# drop if adm2_pcod2 is missing
df_ch_sub = df_ch_sub.dropna(subset=['adm1_5_pcod2'])


# merge df_ch_sub with gdf_ch_shape_combined on adm2_pcod2
gdf_ch_merged_4 = df_ch_sub.merge(gdf_ch_shape_combined, on='adm1_5_pcod2', how='left', indicator=True)

In [19]:
gdf_ch_merged_4 = gdf_ch_merged_4[gdf_ch_merged_4['_merge'] != 'left_only'].drop(columns=['_merge'])

In [20]:
df_ch_remaining = df_ch_remaining[~df_ch_remaining['id'].isin(gdf_ch_merged_4['id'])]

In [21]:
# for gdf_ch_shape_2 and gdf_ch_shape, keep only admin2_pcod2 and geometry, then unique on admin2_pcod2 and append these two geodataframe into one gdf
gdf_ch_shape_sub = gdf_ch_shape[['adm1_pcod2', 'geometry']].drop_duplicates(subset=['adm1_pcod2'])


# drop if adm2_pcod2 is missing
gdf_ch_shape_combined = gdf_ch_shape_sub.dropna(subset=['adm1_pcod2'])


# keep adm0_name, adm2_name, adm2_pcod2, reference_year, reference_label, population, phase_class, phase1, phase2, phase3, phase4, phase5, phase35
df_ch_sub = df_ch_remaining[['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# drop if adm2_pcod2 is missing
df_ch_sub = df_ch_sub.dropna(subset=['adm1_pcod2'])


# merge df_ch_sub with gdf_ch_shape_combined on adm2_pcod2
gdf_ch_merged_5 = df_ch_sub.merge(gdf_ch_shape_combined, on='adm1_pcod2', how='left', indicator=True)

In [22]:
gdf_ch_merged_5 = gdf_ch_merged_5[gdf_ch_merged_5['_merge'] != 'left_only'].drop(columns=['_merge'])

In [23]:
df_ch_remaining = df_ch_remaining[~df_ch_remaining['id'].isin(gdf_ch_merged_5['id'])]

In [24]:
# for gdf_ch_shape_2 and gdf_ch_shape, keep only admin2_pcod2 and geometry, then unique on admin2_pcod2 and append these two geodataframe into one gdf
gdf_ch_shape_sub = gdf_ch_shape[['adm0_pcod3', 'geometry']].drop_duplicates(subset=['adm0_pcod3'])


# drop if adm2_pcod2 is missing
gdf_ch_shape_combined = gdf_ch_shape_sub.dropna(subset=['adm0_pcod3'])


# keep adm0_name, adm2_name, adm2_pcod2, reference_year, reference_label, population, phase_class, phase1, phase2, phase3, phase4, phase5, phase35
df_ch_sub = df_ch_remaining[['id','adm0_name', 'adm1_name', 'adm0_pcod3', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# drop if adm2_pcod2 is missing
df_ch_sub = df_ch_sub.dropna(subset=['adm0_pcod3'])


# merge df_ch_sub with gdf_ch_shape_combined on adm2_pcod2
gdf_ch_merged_6 = df_ch_sub.merge(gdf_ch_shape_combined, on='adm0_pcod3', how='left', indicator=True)

In [25]:
gdf_ch_merged_6 = gdf_ch_merged_6[gdf_ch_merged_6['_merge'] != 'left_only'].drop(columns=['_merge'])

In [26]:
df_ch_remaining = df_ch_remaining[~df_ch_remaining['id'].isin(gdf_ch_merged_6['id'])]

In [28]:
print(gdf_ch_merged.columns)

Index(['id', 'adm0_name', 'adm2_name', 'adm3_pcod2', 'reference_year',
       'reference_label', 'population', 'phase_class', 'phase1', 'phase2',
       'phase3', 'phase4', 'phase5', 'phase35', 'geometry'],
      dtype='object')


In [29]:
#rename adm2_name as title, drop adm3_pcod2
gdf_ch_merged = gdf_ch_merged.rename(columns={'adm2_name': 'title'}).drop(columns=['adm3_pcod2'])
gdf_ch_merged_2 = gdf_ch_merged_2.rename(columns={'adm2_name': 'title'}).drop(columns=['adm2_5_pcod2'])
gdf_ch_merged_3 = gdf_ch_merged_3.rename(columns={'adm2_name': 'title'}).drop(columns=['adm2_pcod2'])
gdf_ch_merged_4 = gdf_ch_merged_4.rename(columns={'adm1_name': 'title'}).drop(columns=['adm1_5_pcod2'])
gdf_ch_merged_5 = gdf_ch_merged_5.rename(columns={'adm1_name': 'title'}).drop(columns=['adm1_pcod2'])
gdf_ch_merged_6 = gdf_ch_merged_6.rename(columns={'adm1_name': 'title'}).drop(columns=['adm0_pcod3'])

In [30]:
# append gdf_ch_merged, gdf_ch_merged_2, gdf_ch_merged_3, gdf_ch_merged_4, gdf_ch_merged_5, gdf_ch_merged_6
gdf_ch_final = gpd.GeoDataFrame(pd.concat([gdf_ch_merged, gdf_ch_merged_2, gdf_ch_merged_3, gdf_ch_merged_4, gdf_ch_merged_5, gdf_ch_merged_6], ignore_index=True))

In [31]:
# drop id
gdf_ch_final = gdf_ch_final.drop(columns=['id'])

In [33]:
gdf_ch_final.columns

Index(['adm0_name', 'title', 'reference_year', 'reference_label', 'population',
       'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5',
       'phase35', 'geometry'],
      dtype='object')

In [32]:
gdf_ipc_sub.columns

Index(['id', 'estimated_population', 'population', 'from', 'to', 'anl_id',
       'title', 'overall_phase', 'country', 'year', 'phase3_worse_population',
       'phase1_population', 'phase2_population', 'phase3_population',
       'phase4_population', 'phase5_population', 'geometry'],
      dtype='object')

In [34]:
# for gdf_ch_final, rename reference_year to year, reference_label to from_to, population to estimated_population, phase_class to overall_phase, phase1 to phase1_population, phase2 to phase2_population, phase3 to phase3_population, phase4 to phase4_population, phase5 to phase5_population, phase35 to phase3_worse_population, adm0_name to country

gdf_ch_final = gdf_ch_final.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country'
})



In [36]:
# split from_to into from and to on ' - ', and then add year to the end
gdf_ch_final[['from', 'to']] = gdf_ch_final['from_to'].str.split('-', expand=True)
gdf_ch_final = gdf_ch_final.drop(columns=['from_to'])
gdf_ch_final['from'] = gdf_ch_final['from'] + ' ' + gdf_ch_final['year'].astype(str)

In [37]:

#append gdf_ch_final to gdf_ipc_sub
gdf_ipc_ch_combined = gpd.GeoDataFrame(pd.concat([gdf_ipc_sub, gdf_ch_final], ignore_index=True))

In [68]:
df_ch_remaining_CAF = df_ch_remaining[df_ch_remaining['adm0_name'] == 'Central African Republic']

In [69]:
# for df_ch_remaining_CAF, if adm2_name == 'Arrondissement 1' and adm0_pcod3 == 'CAF', change adm2_pcod2=='CF7111'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 1') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 2') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 3') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 4') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 5') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
# also for 6,7,8
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 6') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 7') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'
df_ch_remaining_CAF.loc[(df_ch_remaining_CAF['adm2_name'] == 'Arrondissement 8') & (df_ch_remaining_CAF['adm0_pcod3'] == 'CAF'), 'adm2_pcod2'] = 'CF711'



In [70]:
# read caf_admin2_em.geojson in C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\CAF_shape\caf_admin2_em.geojson
PATH_CAF_ADMIN2 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\CAF_shape\caf_admin2_em.geojson'
gdf_caf_admin2 = gpd.read_file(PATH_CAF_ADMIN2)

In [71]:
# for df_ch_remaining_CAF adm2_pcod2, remove '0' in the string
df_ch_remaining_CAF['adm2_pcod2'] = df_ch_remaining_CAF['adm2_pcod2'].str.replace('0', '', regex=False)

C:\Users\swl00\AppData\Local\Temp\ipykernel_17616\873975770.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ch_remaining_CAF['adm2_pcod2'] = df_ch_remaining_CAF['adm2_pcod2'].str.replace('0', '', regex=False)


In [72]:
# keep only [['id','adm0_name', 'adm2_name', 'adm2_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
df_ch_remaining_CAF_sub = df_ch_remaining_CAF[['id','adm0_name', 'adm2_name', 'adm2_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# gdf_caf_admin2, keep only ['adm2_pcode', 'geometry']
gdf_caf_admin2_sub = gdf_caf_admin2[['adm2_pcode', 'geometry']]

# rename adm2_pcode to adm2_pcod2
gdf_caf_admin2_sub = gdf_caf_admin2_sub.rename(columns={'adm2_pcode': 'adm2_pcod2'})

# merge df_ch_remaining_CAF_sub with gdf_caf_admin2_sub on adm2_pcod2
gdf_ch_caf_merged = df_ch_remaining_CAF_sub.merge(gdf_caf_admin2_sub, on='adm2_pcod2', how='left', indicator=True)

In [73]:
# see _merge
print(gdf_ch_caf_merged['_merge'].value_counts())

_merge
both          935
left_only       9
right_only      0
Name: count, dtype: int64


In [74]:
# read caf_admin0_em_geojson
PATH_CAF_ADMIN0 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\CAF_shape\caf_admin0_em.geojson'
gdf_caf_admin0 = gpd.read_file(PATH_CAF_ADMIN0)

In [75]:
# append gdf_caf_admin0 geometry to gdf_ch_caf_merged where _merge=='left_only'
gdf_ch_caf_left_only = gdf_ch_caf_merged[gdf_ch_caf_merged['_merge'] == 'left_only'].drop(columns=['geometry', '_merge'])
gdf_caf_admin0_sub = gdf_caf_admin0[['geometry']].copy()
# repeat length of gdf_ch_caf_left_only
gdf_caf_admin0_sub = pd.concat([gdf_caf_admin0_sub]*len(gdf_ch_caf_left_only), ignore_index=True)

gdf_ch_caf_left_only = gdf_ch_caf_left_only.reset_index(drop=True)
gdf_caf_admin0_sub = gdf_caf_admin0_sub.reset_index(drop=True)
gdf_ch_caf_left_only = pd.concat([gdf_ch_caf_left_only, gdf_caf_admin0_sub], axis=1)
# concat back
gdf_ch_caf_merged_final = pd.concat([gdf_ch_caf_merged[gdf_ch_caf_merged['_merge'] != 'left_only'], gdf_ch_caf_left_only], ignore_index=True)

In [76]:
# drop _merge
gdf_ch_caf_merged_final = gdf_ch_caf_merged_final.drop(columns=['_merge'])

In [77]:
# rename columns as before
gdf_ch_caf_merged_final = gdf_ch_caf_merged_final.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country',
    'adm2_name': 'title'
})

# extract from and to
gdf_ch_caf_merged_final[['from', 'to']] = gdf_ch_caf_merged_final['from_to'].str.split('-', expand=True)
gdf_ch_caf_merged_final = gdf_ch_caf_merged_final.drop(columns=['from_to'])
gdf_ch_caf_merged_final['from'] = gdf_ch_caf_merged_final['from'] + ' ' + gdf_ch_caf_merged_final['year'].astype(str)
# append gdf_ch_caf_merged_final to gdf_ipc_ch_combined
gdf_ipc_ch_final = gpd.GeoDataFrame(pd.concat([gdf_ipc_ch_combined, gdf_ch_caf_merged_final], ignore_index=True))

In [78]:
# remove Central African Republic from df_ch_remaining
df_ch_remaining = df_ch_remaining[df_ch_remaining['adm0_name'] != 'Central African Republic']

In [79]:
# remove columns:adm2_5_name, adm2_5_pcod2, adm3_name, adm3_pcod2
df_ch_remaining = df_ch_remaining.drop(columns=['adm2_5_name', 'adm2_5_pcod2', 'adm3_name', 'adm3_pcod2'])

In [81]:
# do the same for Togo
df_ch_remaining_Togo = df_ch_remaining[df_ch_remaining['adm0_name'] == 'Togo']
# read Togo shape file C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\Togo_shape\tgo_admin2_em.geojson
PATH_TOGO_ADMIN2 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\Togo_shape\Shapefiles\tgo_admbnda_adm2_inseed_itos_20210107.shp'
gdf_togo_admin2 = gpd.read_file(PATH_TOGO_ADMIN2)

In [82]:
# keep only [['id','adm0_name', 'adm2_name', 'adm2_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
df_ch_remaining_Togo_sub = df_ch_remaining_Togo[['id','adm0_name', 'adm2_name', 'adm2_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
# gdf_togo_admin2, keep only ['ADM2_PCODE', 'geometry']
gdf_togo_admin2_sub = gdf_togo_admin2[['ADM2_PCODE', 'geometry']]
# rename ADM2_PCODE to adm2_pcod2
gdf_togo_admin2_sub = gdf_togo_admin2_sub.rename(columns={'ADM2_PCODE': 'adm2_pcod2'})
# merge df_ch_remaining_Togo_sub with gdf_togo_admin2_sub on adm2_pcod2
gdf_ch_togo_merged = df_ch_remaining_Togo_sub.merge(gdf_togo_admin2_sub, on='adm2_pcod2', how='left', indicator=True)
# see _merge
print(gdf_ch_togo_merged['_merge'].value_counts())

_merge
both          549
left_only       5
right_only      0
Name: count, dtype: int64


In [83]:
# rename columns as before
gdf_ch_togo_merged = gdf_ch_togo_merged.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country',
    'adm2_name': 'title'
})
# extract from and to
gdf_ch_togo_merged[['from', 'to']] = gdf_ch_togo_merged['from_to'].str.split('-', expand=True)
gdf_ch_togo_merged = gdf_ch_togo_merged.drop(columns=['from_to'])
gdf_ch_togo_merged['from'] = gdf_ch_togo_merged['from'] + ' ' + gdf_ch_togo_merged['year'].astype(str)

In [84]:
# in gdf_ch_togo_merged, extract left_only, and then drop _merge
gdf_ch_togo_left_only = gdf_ch_togo_merged[gdf_ch_togo_merged['_merge'] == 'left_only'].drop(columns=['geometry', '_merge'])

gdf_ch_togo_merged = gdf_ch_togo_merged[gdf_ch_togo_merged['_merge'] != 'left_only'].drop(columns=['_merge'])

In [85]:
# see togo remaining (id in gdf_ch_togo_left_only)
df_ch_Togo_still_remaining = df_ch_remaining_Togo[df_ch_remaining_Togo['id'].isin(gdf_ch_togo_left_only['id'])]

In [86]:
#read adm1 shape for togo
PATH_TOGO_ADMIN1 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\Togo_shape\Shapefiles\tgo_admbnda_adm1_inseed_itos_20210107.shp'
gdf_togo_admin1 = gpd.read_file(PATH_TOGO_ADMIN1)
# keep only ['ADM1_PCODE', 'geometry']
gdf_togo_admin1_sub = gdf_togo_admin1[['ADM1_PCODE', 'geometry']]

# rename ADM1_PCODE to adm1_pcod2
gdf_togo_admin1_sub = gdf_togo_admin1_sub.rename(columns={'ADM1_PCODE': 'adm1_pcod2'})
# keep only [['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
df_ch_Togo_still_remaining_sub = df_ch_Togo_still_remaining[['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]

# merge df_ch_Togo_still_remaining_sub with gdf_togo_admin1_sub on adm1_pcod2
gdf_ch_togo_merged_2 = df_ch_Togo_still_remaining_sub.merge(gdf_togo_admin1_sub, on='adm1_pcod2', how='left', indicator=True)

# see _merge
print(gdf_ch_togo_merged_2['_merge'].value_counts())

_merge
both          5
left_only     0
right_only    0
Name: count, dtype: int64


In [87]:
# rename columns as before
gdf_ch_togo_merged_2 = gdf_ch_togo_merged_2.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country',
    'adm1_name': 'title'
})

# extract from and to
gdf_ch_togo_merged_2[['from', 'to']] = gdf_ch_togo_merged_2['from_to'].str.split('-', expand=True)
gdf_ch_togo_merged_2 = gdf_ch_togo_merged_2.drop(columns=['from_to'])
gdf_ch_togo_merged_2['from'] = gdf_ch_togo_merged_2['from'] + ' ' + gdf_ch_togo_merged_2['year'].astype(str)
# append gdf_ch_togo_merged and gdf_ch_togo_merged_2
gdf_ch_togo_final = gpd.GeoDataFrame(pd.concat([gdf_ch_togo_merged, gdf_ch_togo_merged_2[gdf_ch_togo_merged_2['_merge'] != 'left_only'].drop(columns=['_merge'])], ignore_index=True))

In [88]:
# drop adm2_pcod2 and adm1_pcod2
gdf_ch_togo_final = gdf_ch_togo_final.drop(columns=['adm2_pcod2', 'adm1_pcod2'])

# append gdf_ch_togo_final to gdf_ipc_ch_final
gdf_ipc_ch_final = gpd.GeoDataFrame(pd.concat([gdf_ipc_ch_final, gdf_ch_togo_final], ignore_index=True))

In [89]:
# df_ch_remaining now remove Togo
df_ch_remaining = df_ch_remaining[df_ch_remaining['adm0_name'] != 'Togo']

In [94]:
#now extract Cape verde
df_ch_remaining_CapeVerde = df_ch_remaining[df_ch_remaining['adm0_name'] == 'Cabo Verde']
# for the remaining data, only process admin1
# read Cape Verde shape file C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\CapeVerde_shape\cpv_admin1_em.geojson
PATH_CAPEVERDE_ADMIN1 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\Cape_Verde_shape\cpv_admbnda_adm1_20240607.shp'
gdf_capeverde_admin1 = gpd.read_file(PATH_CAPEVERDE_ADMIN1)


In [95]:

# keep only [['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
df_ch_remaining_CapeVerde_sub = df_ch_remaining_CapeVerde[['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
# gdf_capeverde_admin1, keep only ['ADM1_PCODE', 'geometry']
gdf_capeverde_admin1_sub = gdf_capeverde_admin1[['ADM1_PCODE', 'geometry']]
# rename ADM1_PCODE to adm1_pcod2'
gdf_capeverde_admin1_sub = gdf_capeverde_admin1_sub.rename(columns={'ADM1_PCODE': 'adm1_pcod2'})
# merge df_ch_remaining_CapeVerde_sub with gdf_capeverde_admin1_sub on adm1_pcod2
gdf_ch_capeverde_merged = df_ch_remaining_CapeVerde_sub.merge(gdf_capeverde_admin1_sub, on='adm1_pcod2', how='left', indicator=True)
# see _merge
print(gdf_ch_capeverde_merged['_merge'].value_counts())

_merge
both          220
left_only       0
right_only      0
Name: count, dtype: int64


In [96]:
# drop _merge
gdf_ch_capeverde_merged = gdf_ch_capeverde_merged[gdf_ch_capeverde_merged['_merge'] != 'left_only'].drop(columns=['_merge'])
# rename columns as before
gdf_ch_capeverde_merged = gdf_ch_capeverde_merged.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country',
    'adm1_name': 'title'
})

# extract from and to
gdf_ch_capeverde_merged[['from', 'to']] = gdf_ch_capeverde_merged['from_to'].str.split('-', expand=True)
gdf_ch_capeverde_merged = gdf_ch_capeverde_merged.drop(columns=['from_to'])
gdf_ch_capeverde_merged['from'] = gdf_ch_capeverde_merged['from'] + ' ' + gdf_ch_capeverde_merged['year'].astype(str)
# drop adm1_pcod2
gdf_ch_capeverde_merged = gdf_ch_capeverde_merged.drop(columns=['adm1_pcod2'])
# append gdf_ch_capeverde_merged to gdf_ipc_ch_final
gdf_ipc_ch_final = gpd.GeoDataFrame(pd.concat([gdf_ipc_ch_final, gdf_ch_capeverde_merged], ignore_index=True))

In [97]:
# in gdf_ipc_ch_final, remove adm2_pcod2
gdf_ipc_ch_final = gdf_ipc_ch_final.drop(columns=['adm2_pcod2'])

In [98]:
# for df_ch_remaining, remove Cape Verde
df_ch_remaining = df_ch_remaining[df_ch_remaining['adm0_name'] != 'Cabo Verde']

In [100]:
# now deal with Liberia
df_ch_remaining_Liberia = df_ch_remaining[df_ch_remaining['adm0_name'] == 'Liberia']
# read adm1 shape for Liberia
PATH_LIBERIA_ADMIN1 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\Liberia_Shape\lbr_adm_ocha_20191104_SHP\lbr_admbnda_adm1_ocha_20191104.shp'
gdf_liberia_admin1 = gpd.read_file(PATH_LIBERIA_ADMIN1)

In [101]:
# keep only ['ADM1_PCODE', 'geometry']
gdf_liberia_admin1_sub = gdf_liberia_admin1[['ADM1_PCODE', 'geometry']]
# rename ADM1_PCODE to adm1_pcod2
gdf_liberia_admin1_sub = gdf_liberia_admin1_sub.rename(columns={'ADM1_PCODE': 'adm1_pcod2'})
# keep only [['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
df_ch_remaining_Liberia_sub = df_ch_remaining_Liberia[['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
# merge df_ch_remaining_Liberia_sub with gdf_liberia_admin1_sub on adm1_pcod2
gdf_ch_liberia_merged = df_ch_remaining_Liberia_sub.merge(gdf_liberia_admin1_sub, on='adm1_pcod2', how='left', indicator=True)
# see _merge
print(gdf_ch_liberia_merged['_merge'].value_counts())

_merge
both          106
left_only       0
right_only      0
Name: count, dtype: int64


In [102]:
# rename columns as before
gdf_ch_liberia_merged = gdf_ch_liberia_merged.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country',
    'adm1_name': 'title'
})
# extract from and to
gdf_ch_liberia_merged[['from', 'to']] = gdf_ch_liberia_merged['from_to'].str.split('-', expand=True)
gdf_ch_liberia_merged = gdf_ch_liberia_merged.drop(columns=['from_to'])
gdf_ch_liberia_merged['from'] = gdf_ch_liberia_merged['from'] + ' ' + gdf_ch_liberia_merged['year'].astype(str)
# drop adm1_pcod2
gdf_ch_liberia_merged = gdf_ch_liberia_merged.drop(columns=['adm1_pcod2'])
# drop id
gdf_ch_liberia_merged = gdf_ch_liberia_merged.drop(columns=['id'])
# append gdf_ch_liberia_merged to gdf_ipc_ch_final
gdf_ipc_ch_final = gpd.GeoDataFrame(pd.concat([gdf_ipc_ch_final, gdf_ch_liberia_merged], ignore_index=True))

In [103]:
#gdf_ipc_ch_final drop _merge
gdf_ipc_ch_final = gdf_ipc_ch_final.drop(columns=['_merge'], errors='ignore')

In [106]:
# in gdf_ipc_ch_final, if anal_id is missing, set id is missing as well
gdf_ipc_ch_final.loc[gdf_ipc_ch_final['anl_id'].isna(), 'id'] = None

In [107]:
# remove liberia from df_ch_remaining
df_ch_remaining = df_ch_remaining[df_ch_remaining['adm0_name'] != 'Liberia']

In [108]:
#read Gambia shapefile
df_ch_remaining_Gambia = df_ch_remaining[df_ch_remaining['adm0_name'] == 'Gambia']
PATH_GAMBIA_ADMIN1 = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\Gambia_Shape\gmb_adm_ndma_20220901_SHP\gmb_admbnda_adm1_ndma_20220901.shp'

gdf_gambia_admin1 = gpd.read_file(PATH_GAMBIA_ADMIN1)
# keep only ['ADM1_PCODE', 'geometry']
gdf_gambia_admin1_sub = gdf_gambia_admin1[['ADM1_PCODE', 'geometry']]
# rename ADM1_PCODE to adm1_pcod2
gdf_gambia_admin1_sub = gdf_gambia_admin1_sub.rename(columns={'ADM1_PCODE': 'adm1_pcod2'})
# keep only [['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
df_ch_remaining_Gambia_sub = df_ch_remaining_Gambia[['id','adm0_name', 'adm1_name', 'adm1_pcod2', 'reference_year', 'reference_label', 'population', 'phase_class', 'phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase35']]
# merge df_ch_remaining_Gambia_sub with gdf_gambia_admin1_sub on adm1_pcod2
gdf_ch_gambia_merged = df_ch_remaining_Gambia_sub.merge(gdf_gambia_admin1_sub, on='adm1_pcod2', how='left', indicator=True)
# see _merge
print(gdf_ch_gambia_merged['_merge'].value_counts())

_merge
both          95
left_only      0
right_only     0
Name: count, dtype: int64


In [109]:
# rename columns as before
gdf_ch_gambia_merged = gdf_ch_gambia_merged.rename(columns={
    'reference_year': 'year',
    'reference_label': 'from_to',
    'population': 'estimated_population',
    'phase_class': 'overall_phase',
    'phase1': 'phase1_population',
    'phase2': 'phase2_population',
    'phase3': 'phase3_population',
    'phase4': 'phase4_population',
    'phase5': 'phase5_population',
    'phase35': 'phase3_worse_population',
    'adm0_name': 'country',
    'adm1_name': 'title'
})
# extract from and to
gdf_ch_gambia_merged[['from', 'to']] = gdf_ch_gambia_merged['from_to'].str.split('-', expand=True)
gdf_ch_gambia_merged = gdf_ch_gambia_merged.drop(columns=['from_to'])
gdf_ch_gambia_merged['from'] = gdf_ch_gambia_merged['from'] + ' ' + gdf_ch_gambia_merged['year'].astype(str)
# drop adm1_pcod2
gdf_ch_gambia_merged = gdf_ch_gambia_merged.drop(columns=['adm1_pcod2'])
# drop id
gdf_ch_gambia_merged = gdf_ch_gambia_merged.drop(columns=['id'])
# append gdf_ch_gambia_merged to gdf_ipc_ch_final
gdf_ipc_ch_final = gpd.GeoDataFrame(pd.concat([gdf_ipc_ch_final, gdf_ch_gambia_merged], ignore_index=True))

In [110]:
# for gdf_ipc_ch_final, drop _merge
gdf_ipc_ch_final = gdf_ipc_ch_final.drop(columns=['_merge'], errors='ignore')

In [112]:
# see how many duplicated rows in gdf_ipc_ch_final based on geometry, country, year, from, to
duplicated_rows = gdf_ipc_ch_final.duplicated(subset=['geometry', 'country', 'year', 'from', 'to'], keep=False)

In [113]:
# drop duplicated rows
gdf_ipc_ch_final = gdf_ipc_ch_final[~duplicated_rows]

In [114]:
# for country, it is now a mix of country names of ISO2 and country english names, use pycountry to standardize them to country english names
import pycountry
def standardize_country_name(country):
    try:
        # first try to get by alpha_2 code
        country_obj = pycountry.countries.get(alpha_2=country)
        if country_obj:
            return country_obj.name
        # then try by name
        country_obj = pycountry.countries.get(name=country)
        if country_obj:
            return country_obj.name
        # then try by official_name
        country_obj = pycountry.countries.get(official_name=country)
        if country_obj:
            return country_obj.name
        return country  # return original if not found
    except:
        return country  # return original in case of any error
    
gdf_ipc_ch_final['country'] = gdf_ipc_ch_final['country'].apply(standardize_country_name)

In [115]:
# create ISO2 and ISO3 columns
def get_iso_codes(country_name):
    try:
        country = pycountry.countries.get(name=country_name)
        if country:
            return pd.Series([country.alpha_2, country.alpha_3])
        else:
            return pd.Series([None, None])
    except:
        return pd.Series([None, None])
gdf_ipc_ch_final[['ISO2', 'ISO3']] = gdf_ipc_ch_final['country'].apply(get_iso_codes)

In [116]:
# export to geojson
OUTPUT_PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\gdf_ipc_ch_final.geojson'
gdf_ipc_ch_final.to_file(OUTPUT_PATH, driver='GeoJSON')

In [117]:
# filter if id is missing
gdf_ch = gdf_ipc_ch_final[gdf_ipc_ch_final['id'].isna()]

In [118]:
# drop id and anl_id
gdf_ch = gdf_ch.drop(columns=['id', 'anl_id'])

# export to geojson
OUTPUT_PATH_CH_ONLY = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\gdf_ch.geojson'
gdf_ch.to_file(OUTPUT_PATH_CH_ONLY, driver='GeoJSON')

In [119]:
# count number of values in country column
print(gdf_ipc_ch_final['country'].value_counts())

country
Nigeria                                  6436
Somalia                                  3537
Yemen                                    1744
Central African Republic                 1665
Congo, The Democratic Republic of the    1594
Sudan                                    1592
Chad                                     1479
Niger                                    1310
Mali                                     1117
South Sudan                              1101
Burkina Faso                              967
Benin                                     953
Senegal                                   950
Mozambique                                767
Cameroon                                  660
Zambia                                    619
Afghanistan                               583
Mauritania                                558
Togo                                      552
Ghana                                     511
Guinea                                    479
Haiti                     

In [120]:
# export the count to csv
OUTPUT_PATH_COUNTRY_COUNT = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\gdf_ipc_ch_final_country_count.csv'
gdf_ipc_ch_final['country'].value_counts().to_csv(OUTPUT_PATH_COUNTRY_COUNT)

In [121]:
# count years
print(gdf_ipc_ch_final['year'].value_counts())

year
2023    5547
2024    4874
2022    4824
2020    4779
2021    3521
2019    3154
2018    2360
2017    2005
2025    1244
2016     628
2015     563
2014     508
Name: count, dtype: int64


In [ ]:
import geopandas as gpd
PATH =  r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\gdf_ipc_ch_final.geojson'

# read geojson
gdf_ipc_ch_final = gpd.read_file(PATH)

# save it as shp
OUTPUT_PATH_SHP = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Outcome\gdf_ipc_ch_final.shp'


# 

C:\Users\swl00\AppData\Local\Temp\ipykernel_36256\1397981478.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_ipc_ch_final.to_file(OUTPUT_PATH_SHP)
C:\Users\swl00\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'estimated_population' to 'estimated_'
  ogr_write(
C:\Users\swl00\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'overall_phase' to 'overall_ph'
  ogr_write(
C:\Users\swl00\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'phase3_worse_population' to 'phase3_wor'
  ogr_write(
C:\Users\swl0

FeatureError: Could not add feature to layer at index 1: Attempt to write non-multipoint (GEOMETRYCOLLECTION) geometry to multipoint shapefile.